In [ ]:
import pandas as pd
import numpy as np

In [12]:
teams = ["ATL", "BOS", "BRK", "CHI", "CHO", "CLE", "DAL", "DEN", "DET", "GSW", "HOU", "IND", "LAC", "LAL", "MEM", "MIA", "MIL", "MIN", "NOP", "NYK", "OKC", "ORL", "PHI", "PHO", "POR", "SAC", "SAS", "TOR", "UTA", "WAS"]
year = 2024

players_reduced_df = pd.read_csv(f"procesed_data/season_player_stats_{year}_reduced/all_players.csv", encoding='utf-8')
players_historical_df = pd.read_csv(f"procesed_data/player_mean_stats_reduced.csv", encoding='utf-8')
final_dataset = pd.DataFrame()

"""
th_player1_component1_1, ..., th_player7_componentM_Z, th_bench_usg, ..., ta_player1_component1_1, ..., ta_player7_componentM_Z, ta_bench_usg, ..., th_player1_historic_component1_1, ..., 
th_player7_historic_componentM_Z, ..., ta_player1_historic_component1_1, ..., ta_player7_historic_componentM_Z, team_home_wins (1 o 0)

th es team home (el local) y ta es el visitante (team away)
Te recuerdo como tengo los datos
players_reduced_df:
player,team,game_id,location,opponent,team_score,opponent_score,win,played,mp,PCA1_1,PCA2_1,PCA2_2,PCA2_3,PCA3_1,PCA3_2,PCA3_3,PCA4_1,PCA4_2,PCA5_1,PCA6_1,PCA6_2,PCA6_3
Trae Young,ATL,34815375-cd40-4317-ad44-8717ba11e81c,Away,CHO,110,116,0,1,36.016666666666666,2.631763088870216,3.8944924219131427,-1.5279943459694423,-0.6279994035474706,-1.5451982574123744,-0.18558370852998024,-0.20355966719340315,-0.44628065053866556,0.5016656667767437,7.29075295152267,-0.8029990398778711,1.7643257616964583,-0.5089810166239245
players_historical_df:
Player,PCA1_1,PCA2_1,PCA3_1,PCA3_2,PCA4_1,PCA5_1,PCA5_2
A.J. Green,-0.5447184752036112,-1.44573631084404,-1.554470988536514,-0.049571596522336975,-1.5844411844289013,1.1229404664155183,1.3094277044864937
"""

pca_cols_current = [c for c in players_reduced_df.columns if c.startswith('PCA')]
pca_cols_historic = [c for c in players_historical_df.columns if c.startswith('PCA')]

for game_id, game_df in players_reduced_df.groupby('game_id'):
    row = {}
    row['game_id'] = game_id
    for team, team_df in game_df.groupby('team'):
        # Team home
        if team_df.iloc[0]['location'] == 'Home':
            prefix = "th"
            if team_df.iloc[0]["win"] == 1:
                team_home_wins = 1
            else:
                team_home_wins = 0
            row['team_home_wins'] = team_home_wins
        else:
            prefix = "ta"
        
        team_df = team_df.sort_values(by='mp', ascending=False)
        top5_mp = team_df.head(5)['mp'].sum()
        bench_mp = team_df['mp'].sum() - top5_mp
        row[f"{prefix}_bench_usg"] = bench_mp / top5_mp if top5_mp > 0 else 0
        # Get the 7 players with the most minutes played
        for player_idx in range(min(7, len(team_df))):  # Asegura máximo 7 jugadores
            player = team_df.iloc[player_idx]
            player_count = player_idx + 1
            for col in pca_cols_current:
                row[f"{prefix}_player{player_count}_{col}"] = player[col]
            
            # Stats históricos (con manejo de errores)
            player_name = player['player']
            hist_data = players_historical_df[players_historical_df['Player'] == player_name]
            
            for col in pca_cols_historic:
                if not hist_data.empty:
                    row[f"{prefix}_player{player_count}_historic_{col}"] = hist_data[col].values[0]
                else:
                    row[f"{prefix}_player{player_count}_historic_{col}"] = 0
    final_dataset = pd.concat([final_dataset, pd.DataFrame([row])], ignore_index=True)





# Guardar resultado
final_dataset.to_csv(f"final_data/season_players_dataset_{year}.csv", index=False, encoding='utf-8-sig')